# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and processing a dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described by a [Croissant schema](https://mlcommons.org/croissant) accessible at: 

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load Croissant metadata and explore basic dataset attributes using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version}")
if hasattr(metadata, 'keywords'):
    print(f"Keywords: {metadata.keywords}")
print(f"Published: {metadata.date_published}")
print(f"License: {metadata.license}")

## 2. Data Overview
List record sets, fields, and columns by their Croissant `@id` for further reference.

In [ ]:
# Show available record sets and fields with their @id and names
print("\nAvailable Record Sets:")
for rs in metadata.record_sets:
    print(f"  - RecordSet @id: {rs.id} | Name: {getattr(rs, 'name', 'N/A')}")
    print("    Fields:")
    if hasattr(rs, 'fields'):
        for field in rs.fields:
            print(f"     * Field @id: {field.id} | Name: {getattr(field, 'name', 'N/A')} | DataType: {getattr(field, 'data_type', 'N/A')}")
    if hasattr(rs, 'columns'):
        print("    Columns:")
        for col in rs.columns:
            print(f"     * Column @id: {col.id} | Name: {getattr(col, 'name', 'N/A')}")
    print()

# Collect RecordSet @ids for later extraction
record_set_ids = [rs.id for rs in metadata.record_sets]
print("RecordSet @ids:", record_set_ids)

## 3. Data Extraction
Load tabular data from each record set into DataFrames, referenced by their RecordSet `@id`.

In [ ]:
dataframes = {}
# Example: If you know the RecordSet @id (from above), set it here or extract all
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for RecordSet {rs_id} with shape {df.shape}")
    except Exception as e:
        print(f"Failed to load records for {rs_id}: {e}")

# Example: show available columns for the first record set
if record_set_ids:
    main_rs = record_set_ids[0]
    print("\nColumns in first RecordSet:", list(dataframes[main_rs].columns))
    display(dataframes[main_rs].head())

## 4. Exploratory Data Analysis (EDA)
Process and filter data using specific field `@ids` (refer back to the overview above). Numeric field selection and transformations are demonstrated below.

In [ ]:
# Choose a numeric field for demonstration
# For real analysis, replace with a valid numeric field @id, e.g., 
# numeric_field = '@id_of_numeric_field'
# Example placeholders used below; edit as per actual field ids and dataset:
example_numeric_field = None

# Find the first float/integer field in the main RecordSet as an example
fields = None
for rs in metadata.record_sets:
    if rs.id == main_rs:
        if hasattr(rs, 'fields'):
            fields = rs.fields
            break
if fields:
    for field in fields:
        if getattr(field, 'data_type', None) in ['Float', 'Integer', 'Number']:
            example_numeric_field = field.id
            print(f"Using numeric field @id: {example_numeric_field}, name: {getattr(field, 'name', 'N/A')}")
            break

if example_numeric_field and example_numeric_field in dataframes[main_rs].columns:
    df = dataframes[main_rs]
    # Drop NAs for the selected numeric analysis
    df_num = df.copy().dropna(subset=[example_numeric_field])
    threshold = df_num[example_numeric_field].mean() if len(df_num) else 10

    # Filter for values above the threshold
    filtered_df = df_num[df_num[example_numeric_field] > threshold]
    print(f"Filtered records in {main_rs} where {example_numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{example_numeric_field}_normalized"] = (filtered_df[example_numeric_field] - filtered_df[example_numeric_field].mean()) / filtered_df[example_numeric_field].std()
    print(f"Normalized {example_numeric_field} for filtered records:")
    display(filtered_df[[example_numeric_field, f"{example_numeric_field}_normalized"]].head())

    # Attempt groupby by another field (category)
    group_field = None
    for field in fields:
        if getattr(field, 'data_type', None) == 'Text' and field.id in filtered_df.columns:
            group_field = field.id
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[example_numeric_field].mean().to_frame(name='mean').reset_index()
        print(f"Grouped mean {example_numeric_field} by {group_field}:")
        display(grouped_df.head())
else:
    print("No numeric field found. Please check available fields and edit the field selection above.")

## 5. Visualization
Visualize the distribution of a selected numeric field and compare groups if applicable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if example_numeric_field and example_numeric_field in dataframes[main_rs].columns:
    df = dataframes[main_rs]
    df_num = df.copy().dropna(subset=[example_numeric_field])
    plt.figure(figsize=(8,4))
    sns.histplot(df_num[example_numeric_field], kde=True, bins=30)
    plt.title(f"Distribution of {example_numeric_field}")
    plt.xlabel(example_numeric_field)
    plt.show()
    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=df_num, x=group_field, y=example_numeric_field)
        plt.title(f"{example_numeric_field} by {group_field}")
        plt.show()
else:
    print("Visualization not possible: no numeric field in the DataFrame.")

## 6. Conclusion
This notebook demonstrated how to use the `mlcroissant` library to load, inspect, process, and visualize data from a dataset described by the Croissant schema.

Key steps included:
- Inspecting record sets and field `@id`s.
- Extracting record data dynamically by ID.
- Basic filtering and normalization using numeric fields.
- Grouped aggregation and visualization using field `@id` as references.

This approach is portable across Croissant-compliant datasets. Update field `@id`s as needed for your particular dataset.